# Проект. Пайплайн подготовки данных для классификации товаров

**Автор:** avo_m25_555

**Задача:** аудит → transforms → текст → Dataset → DataLoader → инженерный вердикт.

**Классы:** Handbags, Shirts, Sports Shoes, Watches.  
Ноутбук **монолитный** — без внешних `.py` проекта.


## Чек-лист ТЗ (этапы)

| Этап | Содержание |
|------|------------|
| 1 | Аудит metadata / class_to_idx / путей / изображений / текстов |
| 2 | train vs val/test transforms + визуализация |
| 3 | Очистка текста, словарь только по train, PAD/UNK, encode |
| 4 | Свой Dataset, проверка одного элемента |
| 5 | DataLoader train/val/test, формы батча, визуализация |
| 6 | Финальный инженерный вывод |


## Этап 0. Окружение и путь к `data/`


In [ ]:
# @title 0.1. Зависимости (Colab)
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "torch",
            "torchvision",
            "pillow",
            "matplotlib",
            "pandas",
        ]
    )
print("IN_COLAB =", IN_COLAB)


In [ ]:
# @title 0.2. Импорты, seed, DATA_ROOT
from __future__ import annotations

import random
import re
import shutil
import sys
import zipfile
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

IN_COLAB = "google.colab" in sys.modules

IMAGE_SIZE = 224
MAX_TEXT_LEN = 24
BATCH_SIZE = 8
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
REQUIRED_COLS = [
    "id",
    "split",
    "class_name",
    "label",
    "image_path",
    "productDisplayName",
]


def _is_data_root(p: Path) -> bool:
    return (
        p.is_dir()
        and (p / "metadata.csv").is_file()
        and (p / "class_to_idx.csv").is_file()
        and (p / "images").is_dir()
    )


def _notebook_dir() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "avo_m25_555_goods.ipynb").is_file() or _is_data_root(cwd / "data"):
        return cwd
    for base in [cwd, *cwd.parents]:
        if (base / "avo_m25_555_goods.ipynb").is_file():
            return base
        nested = base / "avo_Goods_rep"
        if (nested / "avo_m25_555_goods.ipynb").is_file():
            return nested
    return cwd


def _find_data_near(notebook_dir: Path) -> Path | None:
    for c in (
        notebook_dir / "data",
        notebook_dir,
        notebook_dir / "avo_Goods_rep" / "data",
        Path("/content/data"),
        Path("/content/fashion_project_dataset"),
        Path("/content/avo_Goods_rep/data"),
    ):
        if _is_data_root(c):
            return c
    return None


def _extract_zip(zpath: Path, dest: Path) -> Path | None:
    if not zpath.is_file():
        return None
    with zpath.open("rb") as f:
        if f.read(2) != b"PK":
            print("Файл не ZIP:", zpath)
            return None
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(dest)
    if _is_data_root(dest):
        return dest
    for child in dest.iterdir():
        if child.is_dir() and _is_data_root(child):
            return child
    for meta in dest.glob("*/metadata.csv"):
        if _is_data_root(meta.parent):
            return meta.parent
    return None


def _pick_local_folder() -> Path | None:
    try:
        import tkinter as tk
        from tkinter import filedialog
    except Exception as e:
        print("Диалог папки недоступен:", e)
        return None
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)
    chosen = filedialog.askdirectory(
        title="Выберите папку data (metadata.csv + class_to_idx.csv + images/)"
    )
    root.destroy()
    return Path(chosen) if chosen else None


def _upload_zip_colab(save_as: Path) -> Path | None:
    from google.colab import files  # type: ignore

    print("Загрузите fashion_project_dataset.zip…")
    uploaded = files.upload()
    if not uploaded:
        return None
    name, raw = next(iter(uploaded.items()))
    save_as.parent.mkdir(parents=True, exist_ok=True)
    target = save_as if str(save_as).lower().endswith(".zip") else save_as / name
    Path(target).write_bytes(raw)
    print("Сохранено:", target)
    return Path(target)


NOTEBOOK_DIR = _notebook_dir()
print("NOTEBOOK_DIR =", NOTEBOOK_DIR)

DATA_ROOT = _find_data_near(NOTEBOOK_DIR)

if DATA_ROOT is None:
    for z in (
        NOTEBOOK_DIR / "fashion_project_dataset.zip",
        NOTEBOOK_DIR / "goods.zip",
        NOTEBOOK_DIR.parent / "Разработка" / "data_goods" / "fashion_project_dataset.zip",
        Path("/content/fashion_project_dataset.zip"),
    ):
        if z.is_file():
            print("Найден zip:", z)
            DATA_ROOT = _extract_zip(z, NOTEBOOK_DIR / "data")
            if DATA_ROOT is not None:
                break

if DATA_ROOT is None:
    print("data/ рядом с ноутбуком не найдена — выберите папку или загрузите zip.")
    if IN_COLAB:
        z = _upload_zip_colab(Path("/content/fashion_project_dataset.zip"))
        if z is not None:
            DATA_ROOT = _extract_zip(z, Path("/content/data"))
    else:
        picked = _pick_local_folder()
        if picked is not None:
            print("Выбрано:", picked)
            if _is_data_root(picked):
                DATA_ROOT = picked
            elif _is_data_root(picked / "data"):
                DATA_ROOT = picked / "data"
            else:
                for z in list(picked.glob("*.zip")):
                    DATA_ROOT = _extract_zip(z, NOTEBOOK_DIR / "data")
                    if DATA_ROOT is not None:
                        break

assert DATA_ROOT is not None and _is_data_root(DATA_ROOT), (
    "Датасет не найден. Нужна папка data/ рядом с ноутбуком "
    "(metadata.csv, class_to_idx.csv, images/) либо выбор папки / Upload zip."
)

print("DATA_ROOT =", DATA_ROOT)
print("SEED / IMAGE_SIZE / MAX_TEXT_LEN / BATCH =", SEED, IMAGE_SIZE, MAX_TEXT_LEN, BATCH_SIZE)


## Этап 1. Аудит и валидация датасета


In [ ]:
# @title 1.1. Загрузка таблиц и обзор
meta_raw = pd.read_csv(DATA_ROOT / "metadata.csv")
c2i = pd.read_csv(DATA_ROOT / "class_to_idx.csv")

missing_cols = [c for c in REQUIRED_COLS if c not in meta_raw.columns]
assert not missing_cols, f"нет обязательных колонок: {missing_cols}"

print("metadata shape:", meta_raw.shape)
print("columns:", meta_raw.columns.tolist())
print("обязательные колонки OK:", REQUIRED_COLS)
print("\nhead:")
print(meta_raw.head(3).to_string(index=False))
print("\nclass_to_idx:")
print(c2i.to_string(index=False))
print("\nsplit values:", sorted(meta_raw["split"].unique().tolist()))
print("class_name values:", sorted(meta_raw["class_name"].unique().tolist()))
print("label values:", sorted(meta_raw["label"].unique().tolist()))


In [ ]:
# @title 1.2. Распределение по split и классам
print("=== split counts ===")
print(meta_raw["split"].value_counts())
print("\n=== class × split ===")
ct = pd.crosstab(meta_raw["class_name"], meta_raw["split"])
print(ct)
print("\nexpected per class: train=210, val=45, test=45")
for split, n in [("train", 210), ("val", 45), ("test", 45)]:
    assert (ct[split] == n).all(), f"bad counts in {split}"
print("Распределение соответствует описанию датасета.")


In [ ]:
# @title 1.3. class_to_idx, пути, изображения, тексты → clean frame
mapping = dict(zip(c2i["class_name"], c2i["label"]))
mismatch_mask = meta_raw.apply(lambda r: mapping.get(r["class_name"]) != r["label"], axis=1)
mismatch = int(mismatch_mask.sum())

missing_paths: list[str] = []
bad_images: list[str] = []
drop_ids: set = set()
widths, heights, modes = [], [], []

for _, row in meta_raw.iterrows():
    p = DATA_ROOT / row["image_path"]
    if not p.is_file():
        missing_paths.append(str(row["image_path"]))
        drop_ids.add(row["id"])
        continue
    try:
        with Image.open(p) as im:
            im.verify()
        with Image.open(p) as im:
            w, h = im.size
            widths.append(w)
            heights.append(h)
            modes.append(im.mode)
            if im.mode not in ("RGB", "RGBA", "L"):
                bad_images.append(str(row["image_path"]))
                drop_ids.add(row["id"])
    except Exception as e:
        bad_images.append(f"{row['image_path']}: {e}")
        drop_ids.add(row["id"])

names = meta_raw["productDisplayName"].fillna("").astype(str)
empty_mask = names.str.strip() == ""
empty_names = int(empty_mask.sum())
short_mask = names.str.strip().str.split().str.len() < 2
short_names = int(short_mask.sum())
name_len = names.str.split().str.len()

drop_ids |= set(meta_raw.loc[mismatch_mask, "id"].tolist())
drop_ids |= set(meta_raw.loc[empty_mask, "id"].tolist())

if missing_paths:
    print("LOG missing files (первые 5):", missing_paths[:5])
if bad_images:
    print("LOG bad images (первые 5):", bad_images[:5])
if empty_names:
    print("LOG empty productDisplayName:", empty_names)
if mismatch:
    print("LOG class_to_idx mismatch rows:", mismatch)

meta = meta_raw[~meta_raw["id"].isin(drop_ids)].copy().reset_index(drop=True)
n_dropped = len(meta_raw) - len(meta)

print(f"class_to_idx mismatch: {mismatch}")
print(f"missing files: {len(missing_paths)}")
print(f"bad images: {len(bad_images)}")
print(f"empty productDisplayName: {empty_names}")
print(f"short names (<2 tokens): {short_names}")
print(f"name tokens: mean={name_len.mean():.1f}, median={name_len.median():.0f}, max={name_len.max()}")
if widths:
    print(
        f"image size: mean={np.mean(widths):.0f}x{np.mean(heights):.0f}, "
        f"min={min(widths)}x{min(heights)}, max={max(widths)}x{max(heights)}, "
        f"unique modes={sorted(set(modes))}"
    )
print(f"dropped rows: {n_dropped} | clean frame: {len(meta)} / {len(meta_raw)}")
print(
    "связь image↔text↔label после фильтра: "
    f"{meta[['image_path','productDisplayName','label']].notna().all().all()}"
)

assert mismatch == 0
assert len(missing_paths) == 0
assert len(bad_images) == 0
assert empty_names == 0
assert len(meta) == len(meta_raw)
print("\nAUDIT PASSED — clean frame готов")


In [ ]:
# @title 1.4. Примеры сырых изображений
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, cls in zip(axes, ["Handbags", "Shirts", "Sports Shoes", "Watches"]):
    row = meta[(meta["split"] == "train") & (meta["class_name"] == cls)].iloc[0]
    img = Image.open(DATA_ROOT / row["image_path"]).convert("RGB")
    ax.imshow(img)
    ax.set_title(f"{cls}\n{row['productDisplayName'][:28]}", fontsize=8)
    ax.axis("off")
plt.suptitle("Этап 1: сырые примеры train")
plt.tight_layout()
plt.show()


### Вывод этапа 1

- `metadata.csv` загружен, обязательные колонки на месте; split/class/label согласованы с описанием.
- Баланс классов: 210 / 45 / 45 на класс в train/val/test.
- `class_name` ↔ `label` ↔ `class_to_idx.csv` без расхождений.
- Все `image_path` существуют, изображения открываются; режимы L/RGB приводим к RGB в Dataset.
- `productDisplayName`: пустых нет; коротких имён (&lt;2 токена) — по факту аудита (см. вывод ячейки).
- Битые пути логируются и отбрасываются в clean frame; на этом датасете dropped=0.
- **Вердикт:** датасет пригоден для пайплайна.


## Этап 2. Подготовка изображений


In [ ]:
# @title 2.1. train / val / test transforms
def build_train_transforms(image_size: int = IMAGE_SIZE) -> transforms.Compose:
    """Resize + Flip + ColorJitter + ToTensor + ImageNet Normalize."""
    return transforms.Compose(
        [
            transforms.Resize((image_size, image_size)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ]
    )


def build_eval_transforms(image_size: int = IMAGE_SIZE) -> transforms.Compose:
    """Только Resize + ToTensor + Normalize."""
    return transforms.Compose(
        [
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ]
    )


train_transforms = build_train_transforms()
val_transforms = build_eval_transforms()
test_transforms = build_eval_transforms()
print(train_transforms)
print("val/test deterministic:", True)


In [ ]:
# @title 2.2. Визуализация до / train-aug / eval
def denorm(t: torch.Tensor) -> np.ndarray:
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    x = (t * std + mean).clamp(0, 1)
    return x.permute(1, 2, 0).numpy()


sample_rows = [
    meta[(meta["split"] == "train") & (meta["class_name"] == c)].iloc[0]
    for c in ["Handbags", "Shirts", "Sports Shoes", "Watches"]
]

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for j, row in enumerate(sample_rows):
    raw = Image.open(DATA_ROOT / row["image_path"]).convert("RGB")
    axes[0, j].imshow(raw)
    axes[0, j].set_title(row["class_name"])
    axes[0, j].axis("off")
    axes[1, j].imshow(denorm(train_transforms(raw)))
    axes[1, j].axis("off")
    axes[2, j].imshow(denorm(val_transforms(raw)))
    axes[2, j].axis("off")

axes[0, 0].set_ylabel("raw")
axes[1, 0].set_ylabel("train aug")
axes[2, 0].set_ylabel("val/test")
plt.suptitle("Этап 2: до / train-aug / eval")
plt.tight_layout()
plt.show()


### Вывод этапа 2

- **train:** Resize(224) + RandomHorizontalFlip + ColorJitter + Normalize — расширяем разнообразие без сильной деформации товара.
- **val/test:** только Resize + Normalize — стабильная оценка без случайности.
- Разделение обязательно: иначе метрики на val/test будут шумными.


## Этап 3. Подготовка текстовых названий


In [ ]:
# @title 3.1. Очистка, словарь (только train), encode
def clean_text(text: str) -> str:
    """lower, убрать пунктуацию, схлопнуть пробелы."""
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s\-']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text: str) -> list[str]:
    return clean_text(text).split()


def build_vocab(texts: list[str], min_freq: int = 1) -> dict[str, int]:
    """Словарь с <PAD>=0 и <UNK>=1 — только по переданным текстам (train)."""
    counts: Counter[str] = Counter()
    for t in texts:
        counts.update(tokenize(t))
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for tok, c in sorted(counts.items()):
        if c >= min_freq and tok not in vocab:
            vocab[tok] = len(vocab)
    return vocab


def encode_text(text: str, vocab: dict[str, int], max_len: int = MAX_TEXT_LEN) -> list[int]:
    """Токены → индексы фиксированной длины (pad / truncate)."""
    ids = [vocab.get(tok, vocab["<UNK>"]) for tok in tokenize(text)]
    ids = ids[:max_len]
    ids = ids + [vocab["<PAD>"]] * (max_len - len(ids))
    return ids


train_df = meta[meta["split"] == "train"].copy()
val_df = meta[meta["split"] == "val"].copy()
test_df = meta[meta["split"] == "test"].copy()

vocab = build_vocab(train_df["productDisplayName"].tolist(), min_freq=1)
print(f"VOCAB_SIZE={len(vocab)} | MAX_TEXT_LEN={MAX_TEXT_LEN}")
print("special tokens:", {"<PAD>": vocab["<PAD>"], "<UNK>": vocab["<UNK>"]})


In [ ]:
# @title 3.2. Примеры: имя → токены → индексы
examples = train_df["productDisplayName"].head(3).tolist()
examples.append("zzz unknown brand gadget xyz")

rows = []
for name in examples:
    toks = tokenize(name)
    ids = encode_text(name, vocab, MAX_TEXT_LEN)
    rows.append(
        {
            "productDisplayName": name,
            "tokens": " ".join(toks),
            "n_tokens": len(toks),
            "ids_head": ids[:12],
            "len_ids": len(ids),
        }
    )
print(pd.DataFrame(rows).to_string(index=False))
assert all(len(encode_text(n, vocab)) == MAX_TEXT_LEN for n in examples)
print("Все последовательности длины", MAX_TEXT_LEN)


### Вывод этапа 3

- Очистка: lower + простая пунктуация + пробелы — бренд/тип/цвет сохраняются.
- Словарь **только по train**; спецтокены `<PAD>`, `<UNK>`.
- `MAX_TEXT_LEN=24`: pad / truncate → фиксированная длина для батчинга.


## Этап 4. Собственный PyTorch Dataset


In [ ]:
# @title 4.1. Класс FashionDataset
class FashionDataset(Dataset):
    """Мультимодальный датасет: image + text ids + label."""

    def __init__(
        self,
        root: str | Path,
        frame: pd.DataFrame,
        image_transforms: transforms.Compose,
        vocab: dict[str, int],
        max_text_len: int = MAX_TEXT_LEN,
    ):
        self.root = Path(root)
        self.frame = frame.reset_index(drop=True)
        self.image_transforms = image_transforms
        self.vocab = vocab
        self.max_text_len = max_text_len

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, idx: int) -> dict:
        row = self.frame.iloc[idx]
        path = self.root / row["image_path"]
        image = Image.open(path).convert("RGB")
        image_t = self.image_transforms(image)
        name = str(row["productDisplayName"])
        text_ids = torch.tensor(
            encode_text(name, self.vocab, self.max_text_len), dtype=torch.long
        )
        return {
            "image": image_t,
            "text": text_ids,
            "label": int(row["label"]),
            "image_path": str(row["image_path"]),
            "product_name": name,
            "class_name": str(row["class_name"]),
        }


train_ds = FashionDataset(DATA_ROOT, train_df, train_transforms, vocab)
val_ds = FashionDataset(DATA_ROOT, val_df, val_transforms, vocab)
test_ds = FashionDataset(DATA_ROOT, test_df, test_transforms, vocab)
print("sizes train/val/test:", len(train_ds), len(val_ds), len(test_ds))


In [ ]:
# @title 4.2. Проверка одного элемента
item = train_ds[0]
print("keys:", sorted(item.keys()))
print("image type/shape:", type(item["image"]), tuple(item["image"].shape))
print("text type/shape:", type(item["text"]), tuple(item["text"].shape))
print("label:", item["label"], type(item["label"]))
print("class_name:", item["class_name"])
print("product_name:", item["product_name"])
print("image_path:", item["image_path"])

assert item["image"].ndim == 3 and item["image"].shape[0] == 3
assert item["image"].shape[1] == IMAGE_SIZE and item["image"].shape[2] == IMAGE_SIZE
assert item["text"].numel() == MAX_TEXT_LEN
assert item["class_name"] in set(c2i["class_name"])
print("ONE ITEM OK")


### Вывод этапа 4

- `FashionDataset` возвращает словарь `image` / `text` / `label` (+ путь и имя).
- Один элемент проверен: image `[3,224,224]`, text `[24]`, label int.
- Связь изображение ↔ название ↔ класс сохранена.


## Этап 5. DataLoader и проверка батчей


In [ ]:
# @title 5.1. DataLoader + формы батчей train/val/test
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

expected_img = (BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE)
expected_txt = (BATCH_SIZE, MAX_TEXT_LEN)
expected_lab = (BATCH_SIZE,)

batches = {}
for name, loader in [("train", train_loader), ("val", val_loader), ("test", test_loader)]:
    b = next(iter(loader))
    batches[name] = b
    print(f"=== {name} ===")
    print("  image:", tuple(b["image"].shape))
    print("  text:", tuple(b["text"].shape))
    print("  label:", tuple(b["label"].shape))
    assert tuple(b["image"].shape) == expected_img
    assert tuple(b["text"].shape) == expected_txt
    assert tuple(b["label"].shape) == expected_lab

batch = batches["train"]
print("BATCH SHAPES OK — train/val/test готовы к подаче в модель")


In [ ]:
# @title 5.2. Визуализация train-батча
idx2class = dict(zip(c2i["label"], c2i["class_name"]))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = axes.ravel()
for i in range(min(8, BATCH_SIZE)):
    img = denorm(batch["image"][i])
    lab = int(batch["label"][i])
    name = batch["product_name"][i]
    axes[i].imshow(img)
    axes[i].set_title(f"{idx2class[lab]}\n{name[:28]}", fontsize=8)
    axes[i].axis("off")
plt.suptitle("Этап 5: train batch")
plt.tight_layout()
plt.show()


### Вывод этапа 5

- Батчи train/val/test: `[8,3,224,224]`, `[8,24]`, `[8]` — формы проверены на каждом split.
- Визуально подписи классов согласованы с картинками.
- **Вердикт:** батчи корректны и готовы к обучению классификатора.


## Этап 6. Финальный инженерный вывод

### Что сделано

1. **Аудит:** таблицы, баланс классов, пути, RGB/режимы, тексты; clean frame при проблемах.
2. **Изображения:** train — aug; val/test — детерминированные transforms; визуальный контроль.
3. **Текст:** lower/пунктуация; словарь только train; `<PAD>`/`<UNK>`; длина 24.
4. **Dataset:** свой `FashionDataset`, один элемент проверен.
5. **DataLoader:** три лоадера; формы батча на всех split; визуализация train-батча.

### Почему пайплайн готов

- Воспроизводим (`SEED=42`), пути и метки согласованы.
- Формы тензоров стандартны для CNN + текстовой головы.
- Аугментации не ломают семантику товара.

### Ограничения и дальше

- Простая whitespace-токенизация без BPE.
- Weak aug без сильных поворотов/кропов (витринные фото).
- **Дальше:** image-only классификатор, затем multimodal (image+text), сравнение accuracy на val/test.


## Чек-лист самопроверки (ТЗ)

- [x] Ноутбук сверху вниз; этапы 1–6
- [x] metadata загружен; обязательные колонки; split/class/label
- [x] Распределение по классам и разбиениям
- [x] class_to_idx согласован
- [x] image_path проверены; битые → log + drop (clean frame)
- [x] Изображения открываются, RGB; размер/mode
- [x] productDisplayName: пустые/короткие
- [x] Текст очищен; словарь только train; PAD/UNK; фикс. длина
- [x] transforms train ≠ val/test; aug только train
- [x] Dataset + один элемент
- [x] DataLoader train/val/test; формы батча; визуализация
- [x] Выводы после этапов + финальный вердикт
